In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
from pathlib import Path

# Ruta raíz del proyecto en Drive
PROJECT_PATH = Path("/content/drive/MyDrive/proyecto_AAIII_02_diffusion_models")

# Carpetas principales
NOTEBOOKS_DIR = PROJECT_PATH / "notebooks"
SRC_DIR = PROJECT_PATH / "src"
CHECKPOINTS_DIR = PROJECT_PATH / "checkpoints"
RESULTS_DIR = PROJECT_PATH / "results"

# Añadir src al path para poder importar los .py
sys.path.insert(0, str(SRC_DIR))

# Crear carpetas si no existen
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_PATH =", PROJECT_PATH)
print("SRC_DIR =", SRC_DIR)
print("CHECKPOINTS_DIR =", CHECKPOINTS_DIR)
print("RESULTS_DIR =", RESULTS_DIR)

PROJECT_PATH = /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models
SRC_DIR = /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/src
CHECKPOINTS_DIR = /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints
RESULTS_DIR = /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/results


In [3]:

from functools import partial

import torch
from torch.optim import Adam
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
from tqdm.notebook import trange


from score_model import ScoreNet
from diffusion_utilities import plot_image_grid
from ou_utils import (
    cosine_beta_schedule,
    cosine_beta_integral,
    ou_sigma_t,
    build_ou_diffusion_process,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [4]:
# Reproducibility
_ = torch.manual_seed(123)

# Load MNIST
data = datasets.MNIST(
    root=PROJECT_PATH / "data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Keep only digit 3
digit = 3
indices_digit = torch.where(data.targets == digit)[0]
data_train = Subset(data, indices_digit)

print("Training subset size:", len(data_train))

# DataLoader
batch_size = 64

data_loader = DataLoader(
    data_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=torch.get_num_threads(),
)

print("DataLoader ready.")

Training subset size: 6131
DataLoader ready.


In [5]:
# Cosine beta schedule (VP / OU)
beta_schedule = partial(cosine_beta_schedule, s=0.008)
beta_integral = partial(cosine_beta_integral, s=0.008)

# Marginal std function
marginal_prob_std = partial(ou_sigma_t, beta_integral=beta_integral)

print("Cosine schedule ready.")

Cosine schedule ready.


In [6]:
# Build OU diffusion process with cosine schedule
diffusion_process = build_ou_diffusion_process(
    beta_schedule=beta_schedule,
    beta_integral=beta_integral,
)

# Build score model
score_model = torch.nn.DataParallel(
    ScoreNet(
        marginal_prob_std=marginal_prob_std
    )
).to(device)

# Optimizer
learning_rate = 1.0e-3
optimizer = Adam(score_model.parameters(), lr=learning_rate)

print("Model and optimizer ready.")

Model and optimizer ready.


In [7]:
checkpoint_dir = CHECKPOINTS_DIR
checkpoint_dir.mkdir(parents=True, exist_ok=True)

n_epochs = 200
save_epochs = [10, 20, 50, 100, 200]

best_loss = float("inf")
loss_history = []

best_checkpoint_path = checkpoint_dir / "ou_mnist_digit3_cosine_best.pth"

tqdm_epoch = trange(n_epochs)

for epoch in tqdm_epoch:
    score_model.train()
    avg_loss = 0.0
    num_items = 0

    for x, _ in data_loader:
        x = x.to(device)

        loss = diffusion_process.loss_function(score_model, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        avg_loss += loss.item() * x.shape[0]
        num_items += x.shape[0]

    epoch_loss = avg_loss / num_items
    loss_history.append(epoch_loss)

    tqdm_epoch.set_description(f"Average Loss: {epoch_loss:8.5f}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(score_model.state_dict(), best_checkpoint_path)

    if (epoch + 1) in save_epochs:
        epoch_checkpoint_path = checkpoint_dir / f"ou_mnist_digit3_cosine_epoch_{epoch + 1}.pth"
        torch.save(score_model.state_dict(), epoch_checkpoint_path)
        print(f"Saved checkpoint at epoch {epoch + 1}: {epoch_checkpoint_path}")

print("Training finished.")
print("Best loss:", best_loss)
print("Best checkpoint:", best_checkpoint_path)

  0%|          | 0/200 [00:00<?, ?it/s]

Saved checkpoint at epoch 10: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_epoch_10.pth
Saved checkpoint at epoch 20: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_epoch_20.pth
Saved checkpoint at epoch 50: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_epoch_50.pth
Saved checkpoint at epoch 100: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_epoch_100.pth
Saved checkpoint at epoch 200: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_epoch_200.pth
Training finished.
Best loss: 27.60386024752598
Best checkpoint: /content/drive/MyDrive/proyecto_AAIII_02_diffusion_models/checkpoints/ou_mnist_digit3_cosine_best.pth
